# EchoAlign quick start

This notebook loads EchoAlign, reads one echocardiography clip, computes its embeddings, and scores a few findings zero-shot from text prompts.

Before you run it:

1. Install the packages in `requirements.txt`.
2. Download `EchoAlign.pt` into `weights/`.
3. In the first code cell, set the `echo_path` variable to any echo clip. Supported formats are DICOM (`.dcm`), video (`.avi`, `.mp4`), NIfTI (`.nii`, `.nii.gz`), a single image, or a folder of frame images.

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import pandas as pd

from echoalign.loader import load_echo
from echoalign.model import EchoAlign

echo_path = "path/to/your/echo.dcm"
weights_path = "../weights/EchoAlign.pt"

## Load the model

The weights file holds the video encoder, the study transformer, and the text encoder. A GPU is used when available.

In [ ]:
model = EchoAlign(weights_path)
print("Device:", model.device)

## Load the echo

`load_echo` resamples the clip to 24 fps when the file stores its frame rate, reads 16 consecutive frames from the first frame, and returns a tensor of shape `[16, 3, 224, 224]`. DICOM frames are cropped to the ultrasound region and padded to a square first.

In [ ]:
clip = load_echo(echo_path)
print("Input shape:", tuple(clip.shape))

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, t in zip(axes, [0, 5, 10, 15]):
    ax.imshow((clip[t].permute(1, 2, 0).numpy() * 0.5 + 0.5).clip(0, 1))
    ax.set_title(f"Frame {t}")
    ax.axis("off")
plt.show()

## Compute the embeddings

EchoAlign returns 13 study vectors of 512 numbers: one global vector and one vector for each of 12 anatomical structures. A study can have many clips; this example uses one. To embed a full study, pass all of its clips to `encode_study`.

In [ ]:
study = model.encode_study([clip])
print("Study embedding shape:", study.shape)

For linear probes, reduce the 13 vectors to 832 numbers: 64 principal components per vector, fitted on the private cohort's training studies and stored in the weights file.

In [ ]:
features = model.pca_embedding(study)
print("PCA embedding shape:", features.shape)

## Zero-shot scoring

Each finding has positive and negative text prompts. The score is the mean cosine similarity between the study vector and the positive prompts, minus the mean cosine similarity with the negative prompts. Each finding uses the vector of its own anatomical structure.

A higher score means the echo is closer to the positive description. Scores rank studies; they are not probabilities, and no fixed threshold separates positive from negative across sites. The prompts below are examples; edit them or add your own.

In [ ]:
findings = {
    "Aortic stenosis": ("aortic_valve",
        ["Severe aortic stenosis.", "Moderate aortic stenosis with a calcified aortic valve."],
        ["No aortic stenosis.", "Normal aortic valve."]),
    "Mitral regurgitation": ("mitral_valve",
        ["Severe mitral regurgitation.", "Moderate mitral regurgitation."],
        ["No mitral regurgitation.", "Trace mitral regurgitation."]),
    "Reduced LV systolic function": ("left_ventricle",
        ["Severely reduced left ventricular systolic function.", "Moderately reduced left ventricular ejection fraction."],
        ["Normal left ventricular systolic function.", "Normal left ventricular ejection fraction."]),
    "Dilated left atrium": ("left_atrium",
        ["The left atrium is severely dilated.", "Moderate left atrial enlargement."],
        ["Normal left atrial size.", "The left atrium is normal in size."]),
    "Dilated right ventricle": ("right_ventricle",
        ["The right ventricle is severely dilated.", "Moderate right ventricular dilation."],
        ["Normal right ventricular size.", "The right ventricle is normal in size."]),
    "Pericardial effusion": ("pericardium",
        ["Moderate pericardial effusion.", "Large pericardial effusion."],
        ["No pericardial effusion.", "There is no pericardial effusion."]),
}

rows = []
for name, (structure, positive, negative) in findings.items():
    score = model.zero_shot(study, positive, negative, structure=structure)
    rows.append({"finding": name, "vector used": structure, "score": round(score, 4)})
pd.DataFrame(rows).sort_values("score", ascending=False)